# Waiting times: demo

Concept: `overview/Single Level Visit Mappings.md` (outside this repo), section "Waiting Times" with the design decisions DD-W1 to DD-W7, and AS4 (order of events with the same timestamp).
Code: package `ocelescope_module_location_based_ocpm.waiting_times` (modules `scopes`, `waiting`, `durations`, `pipeline`); this notebook only imports and calls their functions.
Further detail: the docstrings (e.g. `help(time_pieces)`) and `docs/resources/waiting-times.md`.

Waiting times are measured in a **scope**: by default each visit at $hl \in HL$ (the visits come from `visits.ipynb`), or, without visits (DD-W6), the whole trace of each handling unit.

What the notebook does:
1. loads the running example (Level 3) and the small edge-case example, and derives their maximal visits,
2. cuts every visit into waiting, storage and remaining time, step by step,
3. does the same **without visits** (whole traces) and compares,
4. runs everything with one call (`analyse_waiting_times`) and shows its options (DD-W1 to DD-W5),
5. goes through the edge cases of waiting times on the small example.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/waiting_times.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from datetime import timedelta
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA, load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import location_context
from ocelescope_module_location_based_ocpm.visits.maximal import add_trace_positions
from ocelescope_module_location_based_ocpm.visits.pipeline import analyse_visits
from ocelescope_module_location_based_ocpm.waiting_times.pipeline import analyse_waiting_times
from ocelescope_module_location_based_ocpm.waiting_times.scopes import scope_relations, trace_scopes, visit_scopes
from ocelescope_module_location_based_ocpm.waiting_times.waiting import (
    add_processing,
    add_storage_removals,
    simultaneous_processing,
    time_pieces,
    waiting_summary,
    waiting_times,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"
EDGE = ROOT / "data" / "visits_example"

# Show whole frames.
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Inputs

### Running example

The location mapping of Level 3, as saved by `location_derivation.ipynb` (`load_level_mapping(folder, level)`), and the location context from `data/running_example/inputs/locations.csv` (`location_context(table)`, see `trace_figure.ipynb`, section 1): $agg$ maps A1, A2 → A and B1 → B; T is not mapped, so it is its own high-level location; $V = \{T\}$, storage units $W = \{B1\}$.

The excursion tolerance is $\kappa_{max} = 2$, as in the note's figure of the maximal visits.

Processing operations: by default $P = T$ (the transformation events: e1, e3, e15); the example of the note uses $P = \{e1, e3, e4, e15\}$ (cast slab, roll, label, cut).
The maximal visits come from `analyse_visits` (see `visits.ipynb`); its `relations` carry the trace positions the visits refer to.

In [ ]:
LEVEL, KAPPA = 3, 2
mapping = load_level_mapping(DERIVED, LEVEL)
context = location_context(pl.read_csv(EXAMPLE / "inputs" / "locations.csv"))
P_NOTE = ["e1", "e3", "e4", "e15"]
visit_analysis = analyse_visits(mapping, "loc", context.agg, context.vehicles, kappa=KAPPA)
visit_analysis.visits.select("visit_id", "hu", "high_level", "first_event", "last_event", "exit_event", "duration")

### Small edge-case example

`data/visits_example/` holds a small example made for these notebooks (generated by `make_visits_example.py`, the single source of its files): eight handling units h1 to h8 whose traces cover the edge cases.
The relations are in the format of a saved level mapping (read with the same schema `MAPPING_SCHEMA`); `loc` is given directly, as if derived.

Location context: A1, A2, the **crane C1** (a vehicle mapped *into* A) and the **storage units S1, S2** lie in hall A, B1 and the storage unit B2 in hall B, and the **truck T** is not mapped (a vehicle with its own high-level location); $V = \{C1, T\}$, $W = \{S1, S2, B2\}$.
Processing operations $P$ are given explicitly in `processing_events.csv`; three of them are movements ($P \cap M \neq \emptyset$).
The example is meant for $\kappa_{max} = 1$.

| handling unit | edge cases of waiting times (section 5) |
|---|---|
| h1 | waiting time after an entry by processing |
| h2 | a visit without processing (all remaining time) |
| h5 | entry without processing, processing on a vehicle inside $hl$, processing during an excursion, processing movements as exit of one visit and entry of the next, trace end |
| h6 | storage units: a move between storage units (no removal), a removal, a processing operation that removes from storage (DD-W1), a removal outside A during an excursion (DD-W2) |
| h7 | events with the same timestamp (AS4): a removal by a movement before a processing operation (waits 0, DD-W4), a processing operation before another event |
| h8 | two processing operations at the same timestamp (RQ5 violated): flagged, ordered by event id |

In [ ]:
EDGE_KAPPA = 1
edge_relations = pl.read_csv(EDGE / "relations.csv", schema=MAPPING_SCHEMA)
edge_context = location_context(pl.read_csv(EDGE / "locations.csv"))
edge_processing = pl.read_csv(EDGE / "processing_events.csv")["event"].to_list()
print(edge_context)
print("P =", edge_processing)
edge_visits = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
)
edge_relations

## 2. Waiting times per visit, step by step

Concept ("Waiting Times"): the waiting time before a processing operation is the time since the handling unit last became ready for it, the *latest* of
- its **entry** into the visited $hl$ (it was brought there to be processed),
- its last **removal from a storage unit** (time in, or on the way to, storage is deliberately parked, not waiting; DD-W5), and
- the **previous processing operation**.

For a visit $\nu = (hu, hl, \sigma)$, $hl \in HL$, with $\sigma$ ordered by $seq$ (AS4):
- **processing positions** $Q(\nu)$: positions $j$ with $\sigma[j] \in P$ and $\widehat{hl}_{\sigma[j],hu} = hl$ (processing at $hl$ only);
- **storage removals** $X_W(\nu)$: positions $j \in \{2, \dots, n\}$ with $loc_{\sigma[j-1]} \in W$ and $loc_{\sigma[j]} \notin W$, anywhere in the visit (DD-W2);
- **clock events** $c = \langle c_1, \dots, c_s \rangle$: the positions $\{1\} \cup Q(\nu) \cup X_W(\nu)$ in increasing order;
- the waiting time of the processing operation at $c[l] \in Q(\nu)$ is $w = time(\sigma[c[l]]) - time(\sigma[c[l-1]])$, and 0 if $l = 1$ (the visit starts with it).

The implementation cuts the visit at all clock events and its exit: every piece is labelled by the event ending it, **waiting** (a processing operation), **storage** (a removal by another event) or **remaining** (the exit); they add up to the visit duration.
Timestamps are completion times (DD-W3), so a waiting time includes the processing time of its own operation (section 4 for deducting a known duration).
$P$ is a separate input; if none is given, the transformation events $T$ are used. Since $loc$ is the location after the event, a processing movement from $hl$ to $hl'$ counts at $hl'$.

### Scopes: `visit_scopes(visits)` and `scope_relations(relations, scopes, location="loc")`

The scopes are the visits at $hl \in HL$; visits at ⊥ get no waiting times (DD-W7: ⊥ is not a place).
`scope_relations` gives one row per scope and relation of its $\sigma$ (as `visit_relations`), with `at_location` = the relation counts for the scope ($\widehat{hl} = hl$).

| parameter | meaning | default |
|---|---|---|
| `visits` | e.g. `VisitAnalysis.visits` | |
| `relations` | the relations the visits were derived from (`VisitAnalysis.relations`, with positions) | |
| `scopes` | output of `visit_scopes` (or `trace_scopes`, section 3) | |
| `location` | column of $loc$ | `"loc"` |

In [ ]:
scopes = visit_scopes(visit_analysis.visits)
srel = scope_relations(visit_analysis.relations, scopes)
print(f"{visit_analysis.visits.height} visits, {scopes.height} scopes (the ⊥ visit of c5 is left out)")
srel.filter(pl.col("visit_id") == 1).select("visit_id", "event", "position", "loc", "high_level", "visit_high_level", "at_location")

### Processing operations: `add_processing(visit_relations, processing=None)`

Adds `is_processing_event` ($e \in P$, wherever it happens) and `is_processing` ($e \in P$ at the visit's $hl$).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `scope_relations` | |
| `processing` | $P$ as event ids | `None` = $T$ (column `is_transformation`) |

Running example with the default ($P = T = \{e1, e3, e15\}$): c1's visit 1 to A has one processing operation, the roll e3 (its entry).

In [ ]:
srel = add_processing(srel)
srel.filter(pl.col("visit_id") == 1).select("visit_id", "event", "activity", "position", "loc", "high_level", "at_location", "is_processing_event", "is_processing")

### Storage removals: `add_storage_removals(visit_relations, storage_units=(), location="loc")`

Adds `is_storage_removal`: the relation is not the first of the visit, the previous relation of the handling unit is in a storage unit and this one is not ($loc_{j-1} \in W$, $loc_j \notin W$).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `scope_relations` / `add_processing` | |
| `storage_units` | the set $W$ | empty (no removals) |
| `location` | column of $loc$ | `"loc"` |
| `outside_storage_resets` | DD-W2. `True`: removals from any storage unit count (concept); `False`: only from storage units at the visit's $hl$ | `True` |

Running example, $W = \{B1\}$: c1 is put into storage at e7 (B1, during an excursion of its visit to A) and removed at e8 (onto the truck T); the removal also counts for the visit to A.

In [ ]:
srel = add_storage_removals(srel, context.storage_units)
srel.filter(pl.col("visit_id") == 1).select("visit_id", "event", "position", "loc", "high_level", "is_processing", "is_storage_removal")

### The pieces: `time_pieces(visit_relations, visits, activity="activity", zero_wait_at_removal=False, skip_simultaneous_removal=False, relations=None)`

One row per piece: `visit_id`, `hu`, `visit_high_level`, `piece` (1, 2, … within the scope), `start_position`, `end_position`, `start_event`, `end_event`, `start_activity`, `end_activity`, `start_time`, `end_time`, `label` (waiting, storage, remaining) and `piece_duration`.

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `add_processing` and `add_storage_removals` | |
| `visits` | the scopes (for the exit events) | |
| `activity` | column of the activity, looked up for the start and end event of each piece (left out if missing) | `"activity"` |
| `zero_wait_at_removal` | DD-W1. `True`: a processing operation that is itself a storage removal ends a storage piece and gets a waiting time of 0; `False`: its waiting time includes the storage time (concept) | `False` |
| `skip_simultaneous_removal` | DD-W4. `True`: a removal by another event at the same timestamp as the next processing operation is no clock event, so the operation waits from the clock event before; `False`: that operation waits 0 (concept) | `False` |
| `relations` | all relations, to look up the activities (also of an exit event outside every scope) | `None` = `visit_relations` |

For a waiting piece, `end_activity` is the activity of the processing operation the waiting time belongs to, so waiting times can be grouped by activity directly (section 5.6).

How it works: the clock events of each scope (entry, processing, removals; the exit from the scopes) are sorted by position; each one after the entry ends a piece starting at the previous one (`shift(1)` per scope). A processing operation at the entry (or at a removal with `zero_wait_at_removal`) gets an extra waiting piece of length 0.

Example: c1's visit 1: e3 (roll, entry) waiting 0, storage from e3 to the removal e8 (5 h), remaining from e8 to the exit e13 (5 h); $d = 10$ h.

In [ ]:
pieces = time_pieces(srel, scopes, relations=visit_analysis.relations)
pieces

### Waiting times: `waiting_times(pieces)`

The pieces labelled "waiting", one per processing operation: `end_event` is the processing operation, `start_event` the clock event its waiting time starts at.

In [ ]:
waiting = waiting_times(pieces)
waiting

### Per scope: `waiting_summary(visits, pieces)`

One row per scope: `n_processing` ($\lvert Q(\nu) \rvert$), `n_storage_removals` (removals that end a storage piece), `total_waiting_time`, `total_storage_time` and `remaining_time` (they add up to the duration), and `is_processing_visit` ($Q(\nu) \neq \emptyset$).
Joined to all visits on `visit_id`, the ⊥ visits get null.

In [ ]:
summary = waiting_summary(scopes, pieces)
visit_analysis.visits.select("visit_id", "hu", "high_level", "first_event", "exit_event", "duration").join(summary, on="visit_id", how="left")

The example of the working doc (C.6): with $P = \{e1, e3, e4, e15\}$ (cast slab, roll, label, cut), c1's visit to A ($m_2$) is cut into waiting 0 (e3), waiting 1 h (e4), storage 4 h (until the removal e8) and remaining 5 h (until e13): $0 + 1 + 4 + 5 = 10 = d$.

The note's table: $Q(m_2) = \{1, 2\}$, $X_W(m_2) = \{5\}$, clock events $c = \langle 1, 2, 5 \rangle$.

In [ ]:
pieces_note = time_pieces(add_processing(srel, P_NOTE), scopes, relations=visit_analysis.relations)
pieces_note.filter(pl.col("visit_id") == 1).drop("visit_high_level", "start_time", "end_time")

## 3. Without visits: the whole trace as one scope

DD-W6: waiting times are determined per visit by default, which needs little domain knowledge: the entry into $hl$ is a natural start of waiting, transports between high-level locations are not counted as waiting, short excursions neither restart the clock nor need modelling, and $P$ can be coarse.
The result depends on how the high-level locations are chosen, though (too fine: the clock restarts too often and waiting is underestimated; too coarse: transports count as waiting and it is overestimated).
So passing visits is optional: **without visits**, the whole trace of a handling unit is one scope, $\sigma = otrace(hu)$, without a high-level location. The clock starts at its first event, and only processing operations and (if selected) storage removals reset it; every located processing operation counts (⊥ ones get no waiting time, as DD-W7).
The last piece ends at the end of the trace (its remaining time is a lower bound); `outside_storage_resets` has no effect.

### Scopes: `trace_scopes(relations)`

One scope per handling unit: `visit_id` numbers the handling units, `high_level` is null, `exit_event` is the last event (`ends_trace` true).
The relations need trace positions (`add_trace_positions`, AS4); `scope_relations` then marks every located relation with `at_location`.

In [ ]:
trace_relations = add_trace_positions(mapping, P_NOTE)
tscopes = trace_scopes(trace_relations)
tscopes

The same steps as in section 2 on the trace scopes, with $P$ and $W = \{B1\}$ of the note.
c1's trace e3 … e14: waiting 0 (roll e3, its first event), waiting 1 h (label e4), storage 4 h (until the removal e8), and remaining time until the trace ends (e14, 6 h). With visits (section 2), the remaining 5 h of the visit to A end at its exit e13, and c1's stay on the truck is a visit of its own.

In [ ]:
trel = scope_relations(trace_relations, tscopes)
trel = add_storage_removals(add_processing(trel, P_NOTE), context.storage_units)
trace_pieces = time_pieces(trel, tscopes, relations=trace_relations)
trace_pieces.filter(pl.col("hu") == "c1").drop("visit_high_level", "start_time", "end_time")

Differences per processing operation: with visits, the cut e15 (c2, in B) waits from c2's entry into B (e12, 3 h); without visits, from c2's previous processing operation (label e4 in A, 11 h), so the transport from A to B and the time at B before the cut count as waiting.

In [ ]:
with_visits = waiting_times(pieces_note).select("hu", "end_event", "end_activity", "start_event", pl.col("piece_duration").alias("with_visits"))
without_visits = waiting_times(trace_pieces).select("hu", "end_event", "start_event", pl.col("piece_duration").alias("without_visits"))
with_visits.join(without_visits, on=["hu", "end_event"], how="full", coalesce=True, suffix="_without").sort("hu", "end_event")

## 4. All in one call: `analyse_waiting_times(relations, location="loc", visits=None, processing=None, storage_units=(), storage_removals_reset=True, outside_storage_resets=True, zero_wait_at_removal=False, skip_simultaneous_removal=False, processing_duration=None, duration_unit="s")`

| parameter | meaning | default |
|---|---|---|
| `relations` | with visits: `VisitAnalysis.relations`; without: any relations frame (positions are added by AS4 if missing) | |
| `location` | column of $loc$ | `"loc"` |
| `visits` | the visits to measure in; `None` = whole traces (DD-W6) | `None` |
| `processing` | $P$ as event ids (use the same as for the visits) | `None` = $T$ |
| `storage_units` | the set $W$ | empty |
| `storage_removals_reset` | DD-W5: removals from storage units are clock events | `True` |
| `outside_storage_resets` | DD-W2: also removals from storage units outside $hl$ (no effect without visits) | `True` |
| `zero_wait_at_removal` | DD-W1: a processing operation that is itself a removal waits 0 | `False` |
| `skip_simultaneous_removal` | DD-W4: skip a removal at the same timestamp as the next processing operation | `False` |
| `processing_duration`, `duration_unit` | DD-W3: event attribute with the processing duration (a duration, or a number in "s", "min" or "h"), deducted from the waiting time of its own operation | `None`, `"s"` |

Returns a `WaitingAnalysis` with the (collected) frames `scopes`, `scope_relations`, `pieces`, `waiting_times`, `summary`, and the flags `simultaneous_processing` (RQ5) and `negative_waiting_times` (DD-W3).
Below: with visits and without visits on the running example (P and W of the note).

In [ ]:
analysis = analyse_waiting_times(
    visit_analysis.relations, "loc", visit_analysis.visits, processing=P_NOTE, storage_units=context.storage_units
)
print("same pieces as step by step:", analysis.pieces.equals(time_pieces(add_storage_removals(add_processing(srel.drop("is_processing_event", "is_processing", "is_storage_removal"), P_NOTE), context.storage_units), scopes, relations=visit_analysis.relations)))
analysis_traces = analyse_waiting_times(mapping, "loc", processing=P_NOTE, storage_units=context.storage_units)
print("same pieces as step by step (whole traces):", analysis_traces.pieces.equals(trace_pieces))
visit_analysis.visits.select("visit_id", "hu", "high_level", "duration").join(analysis.summary, on="visit_id", how="left")

In [ ]:
analysis_traces.scopes.select("visit_id", "hu", "first_event", "exit_event", "duration").join(analysis_traces.summary, on="visit_id")

### Option DD-W5: storage removals as clock events (`storage_removals_reset`)

With `storage_removals_reset=False`, $W$ is ignored: only the entry and processing operations reset the clock. c1's visit to A: waiting 0, 1 h, then 9 h remaining time (no storage piece).

In [ ]:
analyse_waiting_times(
    visit_analysis.relations, "loc", visit_analysis.visits, processing=P_NOTE, storage_units=context.storage_units,
    storage_removals_reset=False,
).pieces.filter(pl.col("visit_id") == 1).drop("visit_high_level", "start_time", "end_time")

### Option DD-W3: processing durations (`processing_duration`)

Timestamps are completion times, so a waiting time also contains the processing time of its own operation $\sigma[c[l]]$ (not of the previous one).
If an event attribute holds the processing duration, it is deducted: `net_waiting_time` = waiting time − processing duration (the waiting time itself if the event has no duration). A negative result (the duration is longer than the time since the last clock event) is flagged in `negative_waiting_time` and listed in `negative_waiting_times`, not corrected.
The running example has no such attribute; for the demo, a made-up column gives the roll e3 a duration of 0.5 h and the label e4 one of 2 h: c1's label waited 1 h, so its net waiting time is −1 h (flagged).

In [ ]:
with_durations = visit_analysis.relations.with_columns(
    pl.col("event").replace_strict({"e3": 0.5, "e4": 2.0}, default=None, return_dtype=pl.Float64).alias("duration_h")
)
durations = analyse_waiting_times(
    with_durations, "loc", visit_analysis.visits, processing=P_NOTE, storage_units=context.storage_units,
    processing_duration="duration_h", duration_unit="h",
)
print(durations.negative_waiting_times.select("hu", "end_event", "piece_duration", "processing_duration", "net_waiting_time"))
durations.waiting_times.select("visit_id", "hu", "end_event", "end_activity", "piece_duration", "processing_duration", "net_waiting_time", "negative_waiting_time")

## 5. Edge cases on the small example

The small example with $\kappa_{max} = 1$, the explicit $P$ and $W = \{S1, S2, B2\}$, measured per visit (`edge`); `edge_table` joins the summary to the visits.

In [ ]:
edge = analyse_waiting_times(
    edge_visits.relations, "loc", edge_visits.visits, processing=edge_processing,
    storage_units=edge_context.storage_units,
)
edge_table = edge_visits.visits.join(edge.summary, on="visit_id", how="left")
edge_table.select("visit_id", "hu", "high_level", "first_event", "exit_event", "duration", *edge.summary.drop("visit_id").columns)

### 5.1 Waiting times (h5, h1)

h5: A1 (crane move, entry), A1 (inspect), A1 (**roll** ∈ P), C1 (**weigh on crane** ∈ P), B1 (**paint and drop** ∈ P ∩ M), A2, A2 (label), B1 (**pack and move** ∈ P ∩ M), B1.
- Visit to A (e33–e39): the entry e33 is no processing operation, so $w_1 = 3 - 0 = 3$ h (roll), $w_2 = 4 - 3 = 1$ h (weighing **on the crane C1**, which is in A, so it counts).
- The paint e37 lies in an excursion of the A visit (at B), so it does **not** count for A; it is the entry of its own visit to B (e37) with $w_1 = 0$.
- The processing movement e40 ends at B1, so it counts at B: it is the **exit event** of the A visit ($w_{rem} = 10 - 4 = 6$ h, $d = 10 = 3 + 1 + 6$) and the **entry** of the next visit to B ($w_1 = 0$); that visit ends with the trace, so its remaining time is a lower bound.
- h1 at A: entry by processing (produce e1, $w_1 = 0$), then the grind e10 after 12 h ($w_2 = 12$ h), $w_{rem} = 1$ h.
- h2's visit to A has no processing operation: its whole duration is remaining time; its visits to T are not processing visits either.

In [ ]:
edge.pieces.filter(pl.col("hu").is_in(["h5", "h1"])).drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_table.filter(pl.col("hu") == "h5").select(
    "visit_id", "high_level", "first_event", "exit_event", "ends_trace", "duration",
    "n_processing", "total_waiting_time", "remaining_time",
)

With the default $P = T$ (here only produce e1 and roll e35), h5's A visit has a single waiting time of 3 h and a remaining time of 7 h.

In [ ]:
edge_default = analyse_waiting_times(edge_visits.relations, "loc", edge_visits.visits, processing=None, storage_units=edge_context.storage_units)
edge_default.pieces.filter(pl.col("hu") == "h5").drop("visit_high_level", "start_time", "end_time")

### 5.2 Storage units (h6)

h6 at A: S1 (entry), S2, C1, A1, A1 (**roll** ∈ P), S1, A2 (**unpack and move** ∈ P ∩ M, takes it out of S1), B2 (storage in B, an excursion), A2, A2 (**grind** ∈ P), then the truck (exit).
- e43 moves it from S1 to S2: **no removal** (still in storage), so the clock keeps running;
- e44 takes it out of S2 onto the crane: **storage** piece of 5 h (e42 to e44); the roll e46 then waits 3 h (from the removal, not from the entry);
- e48 is a processing operation that itself removes from S1: its **waiting time is 4 h** (e46 to e48), including the time in S1 (DD-W1);
- e50 takes it out of B2, a storage unit **outside A** (during the excursion): it still restarts the clock (DD-W2), so the grind e51 waits 1 h, not 3 h;
- remaining 2 h until the exit e52: $5 + 3 + 4 + 3 + 1 + 2 = 18$ h $= d$.

With `zero_wait_at_removal=True` (the second table), the 4 h before e48 are storage time instead and e48 gets a waiting time of 0: $5 + 3 + 4 + 0 + 3 + 1 + 2 = 18$ h.
With `outside_storage_resets=False` (the third table), the removal e50 from B2 (in B, outside A) no longer resets the clock, so the grind e51 waits 4 h from e48: $5 + 3 + 4 + 4 + 2 = 18$ h.

In [ ]:
edge.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_zero = analyse_waiting_times(
    edge_visits.relations, "loc", edge_visits.visits, processing=edge_processing,
    storage_units=edge_context.storage_units, zero_wait_at_removal=True,
)
edge_zero.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_inside = analyse_waiting_times(
    edge_visits.relations, "loc", edge_visits.visits, processing=edge_processing,
    storage_units=edge_context.storage_units, outside_storage_resets=False,
)
edge_inside.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_table.filter(pl.col("hu") == "h6").select(
    "visit_id", "high_level", "first_event", "exit_event", "duration",
    "n_processing", "n_storage_removals", "total_waiting_time", "total_storage_time", "remaining_time",
)

### 5.3 Events with the same timestamp (h7, h8)

AS4 orders events with the same timestamp: movements, then processing operations, then the rest, each group by event id; this order is used everywhere (trace positions, visits, shifts, waiting times).
The ids of h7 are chosen against this order, so the event id order and AS4 differ at both ties:
- t = 3: the crane move **e55** takes h7 out of the storage unit S1 and the **grind e54** (∈ P) happens at the same time. AS4 puts the movement first, so e55 is the removal (storage piece of 3 h from the entry e53) and the grind **waits 0 h** from it (DD-W4).
- t = 5: the **roll e57** (∈ P) comes before the label e56 (not in P), so the roll waits 2 h from the grind e54 and the label falls into the remaining time.
- $3 + 0 + 2 + 1 = 6$ h $= d$. With AS4 the grind at t = 3 is at A1 like the move, as RQ4 requires.

With `skip_simultaneous_removal=True` (second table) the removal e55 is skipped, as it is at the same timestamp as the grind: the grind waits **3 h** from the entry, including the time in S1; $3 + 2 + 1 = 6$ h.

h8 has two processing operations at the same timestamp (grind e60 and roll e61 at t = 2), which RQ5 excludes. This is a flag, not a blocking error (`WaitingAnalysis.simultaneous_processing`, function `simultaneous_processing(relations, processing=None)`); AS4 orders them by event id, so the roll waits 0 after the grind (third and fourth table).

In [ ]:
edge_visits.relations.filter(pl.col("hu") == "h7").select("event", "time", "activity", "is_movement", "loc", "tie_rank", "position")

In [ ]:
edge.pieces.filter(pl.col("hu") == "h7").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_skip = analyse_waiting_times(
    edge_visits.relations, "loc", edge_visits.visits, processing=edge_processing,
    storage_units=edge_context.storage_units, skip_simultaneous_removal=True,
)
edge_skip.pieces.filter(pl.col("hu") == "h7").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge.simultaneous_processing

In [ ]:
edge.waiting_times.filter(pl.col("hu") == "h8").drop("visit_high_level", "start_time", "end_time")

### 5.4 ⊥ (h3)

DD-W7: the ⊥ visits of h3 (e19–e20, e22) are no scopes and get no waiting times; in `edge_table` their waiting columns are null.

In [ ]:
edge_table.filter(pl.col("hu") == "h3").select("visit_id", "high_level", "first_event", "last_event", "n_processing", "remaining_time", "is_processing_visit")

### 5.5 Without visits

The same example measured per whole trace. h5: the paint e37 (at B) now counts too and waits 1 h after the weighing e36, and the remaining time runs to the end of the trace (e41).

In [ ]:
edge_traces = analyse_waiting_times(edge_relations, "loc", processing=edge_processing, storage_units=edge_context.storage_units)
edge_traces.pieces.filter(pl.col("hu") == "h5").drop("visit_high_level", "start_time", "end_time")

### 5.6 Waiting times per activity

Every waiting time carries the activity of its processing operation (`end_activity`) and of the clock event it starts at (`start_activity`), so they can be grouped by activity directly, e.g. number, mean and maximum waiting time per processing activity:

In [ ]:
edge.waiting_times.group_by("end_activity").agg(
    pl.len().alias("n"),
    pl.col("piece_duration").mean().alias("mean_waiting_time"),
    pl.col("piece_duration").max().alias("max_waiting_time"),
).sort("end_activity")

### 5.7 Checks

For every scope, waiting, storage and remaining time add up to its duration, with and without visits.

In [ ]:
for name, result in (
    ("running example, visits", analysis),
    ("running example, traces", analysis_traces),
    ("edge-case example, visits", edge),
    ("edge-case example, traces", edge_traces),
):
    table = result.scopes.select("visit_id", "duration").join(result.summary, on="visit_id")
    total = table["total_waiting_time"] + table["total_storage_time"] + table["remaining_time"]
    print(f"{name}: waiting + storage + remaining = d: {total.equals(table['duration'], check_names=False)}")